# Notebook 5 — Glued non-active $D_r$ sweep

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from front_runner import compile_c_code, run_front_simulations
from front_analysis import (
    read_front_file,
    get_run_files,
    fit_front_speeds_for_front_data,
    side_speed_long_table,
    style_axis,
)

project_folder = Path('.').resolve()

c_file = project_folder / 'abm_front_propagation_glued.c'
exe_file = project_folder / 'abm_front_glued_B30_Dr_sweep.exe'

param_base = 'params_brownian_Dr_sweep_glued'
param_file = project_folder / f'{param_base}.txt'
run_table_file = project_folder / f'{param_base}_runs.csv'

data_folder = project_folder / 'data' / param_base
figures_folder = project_folder / 'figures' / param_base
summary_folder = project_folder / 'analysis_summaries' / param_base

for folder in [data_folder, figures_folder, summary_folder]:
    folder.mkdir(parents=True, exist_ok=True)

compile_first = False
run_simulations = False
skip_existing = True
selected_run_ids = None     # Example: [0, 1, 2] for testing only a few runs
max_parallel = 26

print('Project folder:', project_folder)
print('C file:', c_file)
print('Executable:', exe_file)
print('Parameter file:', param_file)
print('Run table:', run_table_file)
print('Data folder:', data_folder)
print('Figures folder:', figures_folder)

## 1. Sweep parameters

In [ ]:
Lx = 80.0
T = 600.0
Ly_values = np.array([0.5, 1.0, 2.0, 4.0], dtype=float)
R_inter_values = np.array([0.10, 0.08, 0.05], dtype=float)

Dr_values = np.array([0.0005, 0.001, 0.002, 0.004, 0.008], dtype=float)

seed_values = np.array([69069, 1234567, 999999], dtype=int)

isolation_buffer_factor = 30.0

p0 = 0.85
q0 = 0.15
r_edge = p0 - q0

v0 = 0.0
Dtheta = 0.002

dt = 0.001
warmup_T = 200.0
rho0 = 750.0
rho_sat = -1.0

front_per_step = int(round(0.2 / dt))
save_per_step = int(round(10.0 / dt))
rho_profile_every_front = 10

nbins_x = 1600

Ns_fixed = 50

def dt_max_for_R_Dr(R, Dr):
    return (0.1 * float(R))**2 / (2.0 * float(Dr))


dt_max_rows = []

for R in R_inter_values:
    for Dr in Dr_values:
        dt_max_rows.append({
            'R_inter': float(R),
            'Dr': float(Dr),
            'dt_max': dt_max_for_R_Dr(R, Dr),
            'current_dt': dt,
            'current_dt_is_safe': dt <= dt_max_for_R_Dr(R, Dr),
        })

dt_max_table = pd.DataFrame(dt_max_rows)

print('Maximum allowed dt for each (R_inter, Dr):')
display(dt_max_table)

global_dt_max = dt_max_table['dt_max'].min()

print()
print(f'Most restrictive dt_max = {global_dt_max:.6g}')
print(f'Current dt = {dt:.6g}')

if dt <= global_dt_max:
    print('Current dt satisfies the Brownian displacement condition for all planned runs.')
else:
    print('WARNING: Current dt is too large for at least one planned run.')

worst_case = dt_max_table.loc[dt_max_table['dt_max'].idxmin()]
print()
print('Most restrictive case:')
display(pd.DataFrame([worst_case]))

### Fixed interaction parameter

## 2. Write the glued parameter file

In [ ]:
base = dict(
    p0=p0,
    q0=q0,
    rho0=rho0,
    v0=v0,
    Dtheta=Dtheta,
    dt=dt,
    T=T,
    Lx=Lx,
    warmup_T=warmup_T,
    rho_sat=rho_sat,
    nbins_x=nbins_x,
    threshold_frac1=0.2,
    threshold_frac2=0.5,
    threshold_frac3=0.8,
    save_per_step=save_per_step,
    front_per_step=front_per_step,
    rho_profile_every_front=rho_profile_every_front,
    isolation_buffer_factor=isolation_buffer_factor,
)

param_order = [
    'run_id', 'seed',
    'p0', 'q0', 'Ns', 'R_inter', 'rho0',
    'save_per_step', 'front_per_step', 'rho_profile_every_front',
    'v0', 'Dr', 'Dtheta', 'dt', 'T',
    'Lx', 'Ly', 'x_init_min', 'x_init_max', 'warmup_T',
    'rho_sat', 'nbins_x', 'threshold_frac1', 'threshold_frac2', 'threshold_frac3',
    'isolation_buffer_factor',
]

runs = []
run_id = 0

for Ly in Ly_values:
    for R in R_inter_values:
        for Dr in Dr_values:
            for seed in seed_values:
                run = dict(base)
                run.update(
                    run_id=run_id,
                    seed=int(seed),
                    Ns=Ns_fixed,
                    R_inter=float(R),
                    Dr=float(Dr),
                    Ly=float(Ly),
                    x_init_min=0.5 * Lx - 0.25,
                    x_init_max=0.5 * Lx + 0.25,
                )
                run['v_fkpp'] = 2.0 * np.sqrt(run['Dr'] * r_edge)
                run['code_type'] = 'glued'
                runs.append(run)
                run_id += 1

run_table = pd.DataFrame(runs)
run_table.to_csv(run_table_file, index=False)

with open(param_file, 'w') as f:
    f.write('# Glued Brownian Dr sweep with B = 30 R_inter\n')
    f.write('# Goal: compare v_sim and v_sim/v_FKPP across Ly and R_inter\n')
    f.write('# Ns = 50 for all R_inter\n\n')
    for run in runs:
        f.write('[run]\n')
        for key in param_order:
            f.write(f'{key} = {run[key]}\n')
        f.write('\n')

print('Wrote:', param_file)
print('Wrote:', run_table_file)
display(run_table.head(12))

## 3. Compile and run

In [ ]:
if compile_first:
    compile_c_code(
        c_file=c_file,
        exe_file=exe_file,
        flags=['-O2', '-std=c99', '-Wall', '-Wextra'],
    )
else:
    print('compile_first = False')

In [ ]:
if run_simulations:
    results = run_front_simulations(
        exe_file=exe_file,
        param_file=param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing,
        required_output='front',
    )
    display(pd.DataFrame(results))
else:
    print('run_simulations = False')

## 4. Fit front speeds

In [ ]:
main_method = 'th_2'
main_fit_window = (0.70, 0.95)

fit_windows = [
    (0.50, 0.90),
    (0.60, 0.90),
    (0.60, 0.95),
    (0.70, 0.90),
    (0.70, 0.95),
    (0.80, 0.95),
]


def window_label(a, b):
    return f'{a:.2f}-{b:.2f}'


def clean_front_data(front_data, exclude_boundary_hit_rows=True):
    n = len(front_data['time'])
    mask = np.isfinite(np.asarray(front_data['time'], dtype=float))
    if exclude_boundary_hit_rows and 'hit_boundary' in front_data:
        mask &= np.asarray(front_data['hit_boundary'], dtype=float) < 0.5

    out = {}
    for key, value in front_data.items():
        arr = np.asarray(value)
        if arr.shape == (n,):
            out[key] = arr[mask]
        else:
            out[key] = value
    return out


def add_threshold_value_aliases(metadata, front_data):
    for i in range(1, 4):
        key = f'threshold_frac{i}'
        if key not in metadata:
            continue

        threshold_label = f'{float(metadata[key]):g}'.replace('.', 'p')

        for side in ['left', 'right']:
            source_col = f'x_{side}_th_{i}'
            target_col = f'x_{side}_th_{threshold_label}'

            if source_col in front_data and target_col not in front_data:
                front_data[target_col] = front_data[source_col]

    return metadata, front_data

def read_front_file_safe(front_file):
    try:
        metadata, front_data = read_front_file(front_file)
    except TypeError as err:
        if 'delim_whitespace' not in str(err):
            raise

        front_file = Path(front_file)
        header_cols = None
        metadata = {}
        with open(front_file, 'r') as f:
            for line in f:
                if not line.startswith('#'):
                    continue
                parts = line[1:].strip().split()
                if parts and parts[0] == 'step':
                    header_cols = parts
                else:
                    for j in range(0, len(parts) - 1, 2):
                        try:
                            metadata[parts[j]] = float(parts[j + 1])
                        except ValueError:
                            metadata[parts[j]] = parts[j + 1]

        if header_cols is None:
            raise ValueError(f'Could not find front-file header in {front_file}')

        df = pd.read_csv(
            front_file,
            comment='#',
            sep=r'\s+',
            header=None,
            names=header_cols,
        )
        front_data = {col: df[col].to_numpy() for col in df.columns}

    return add_threshold_value_aliases(metadata, front_data)


def fit_runs_for_window(run_table, f0, f1):
    fit_rows = []
    missing = []

    for _, run in run_table.iterrows():
        rid = int(run['run_id'])
        _, front_file, _ = get_run_files(data_folder, param_base, rid)
        front_file = Path(front_file)

        if not front_file.exists():
            missing.append(rid)
            continue

        _, front_data = read_front_file_safe(front_file)
        front_data = clean_front_data(front_data, exclude_boundary_hit_rows=True)

        fit = fit_front_speeds_for_front_data(
            front_data,
            methods=threshold_methods,
            fit_start_fraction=f0,
            fit_end_fraction=f1,
        )

        row = run.to_dict()
        row.update(fit)
        row['fit_window'] = window_label(f0, f1)
        row['fit_start_fraction'] = f0
        row['fit_end_fraction'] = f1
        fit_rows.append(row)

    return pd.DataFrame(fit_rows), missing


run_table = pd.read_csv(run_table_file)
fit_df, missing = fit_runs_for_window(run_table, *main_fit_window)

print(f'Main window {window_label(*main_fit_window)}: fitted {len(fit_df)} / {len(run_table)} runs')
if missing:
    print('Missing front files:', missing[:30], '...' if len(missing) > 30 else '')

fit_df.to_csv(summary_folder / 'fit_df_main_window.csv', index=False)
display(fit_df.head())

## 5. Run-level speed summary

In [ ]:
def make_run_level_speed_table(fit_df, group_cols):
    keep_cols = ['run_id'] + list(group_cols) + ['v_fkpp']
    keep_cols = [c for c in keep_cols if c in fit_df.columns]

    speed_long = side_speed_long_table(
        fit_df,
        method=main_method,
        keep_columns=keep_cols,
    )

    speed_long = speed_long.replace([np.inf, -np.inf], np.nan)
    speed_long = speed_long.dropna(subset=['speed', 'v_fkpp'])
    speed_long['speed_over_fkpp'] = speed_long['speed'] / speed_long['v_fkpp']

    run_cols = ['run_id'] + list(group_cols)
    run_level = (
        speed_long
        .groupby(run_cols, as_index=False)
        .agg(
            run_speed=('speed', 'mean'),
            run_speed_over_fkpp=('speed_over_fkpp', 'mean'),
            left_right_std=('speed_over_fkpp', 'std'),
            n_front_sides=('speed_over_fkpp', 'count'),
            v_fkpp=('v_fkpp', 'first'),
        )
    )

    return run_level


def summarize_run_level(run_level, group_cols):
    summary = (
        run_level
        .groupby(group_cols, as_index=False)
        .agg(
            n_runs=('run_speed', 'count'),
            mean_speed=('run_speed', 'mean'),
            speed_std=('run_speed', 'std'),
            mean_speed_over_fkpp=('run_speed_over_fkpp', 'mean'),
            speed_over_fkpp_std=('run_speed_over_fkpp', 'std'),
            v_fkpp=('v_fkpp', 'first'),
        )
    )

    summary['speed_sem'] = summary['speed_std'] / np.sqrt(summary['n_runs'])
    summary['speed_over_fkpp_sem'] = summary['speed_over_fkpp_std'] / np.sqrt(summary['n_runs'])
    summary['delta_from_fkpp'] = summary['mean_speed_over_fkpp'] - 1.0
    summary['abs_delta_from_fkpp'] = np.abs(summary['delta_from_fkpp'])
    summary['v2_measured'] = summary['mean_speed']**2
    summary['v2_fkpp'] = summary['v_fkpp']**2

    return summary


group_cols = ['Ly', 'R_inter', 'Dr']
run_level_main = make_run_level_speed_table(fit_df, group_cols)
summary = summarize_run_level(run_level_main, group_cols)

run_level_main.to_csv(summary_folder / 'run_level_speeds_main_window.csv', index=False)
summary.to_csv(summary_folder / 'summary_main_window.csv', index=False)

display(summary.sort_values(['Ly', 'R_inter', 'Dr']).head(20))

## 8. Parameter ranking: which $(L_y,R_{inter})$ is closest to FKPP?

In [ ]:
if len(summary) == 0:
    print('No summary to rank yet.')
else:
    geometry_ranking = (
        summary
        .groupby(['Ly', 'R_inter'], as_index=False)
        .agg(
            n_Dr=('Dr', 'count'),
            mean_abs_error_to_fkpp=('abs_delta_from_fkpp', 'mean'),
            max_abs_error_to_fkpp=('abs_delta_from_fkpp', 'max'),
            mean_signed_error_to_fkpp=('delta_from_fkpp', 'mean'),
            mean_v_over_fkpp=('mean_speed_over_fkpp', 'mean'),
        )
        .sort_values(['mean_abs_error_to_fkpp', 'max_abs_error_to_fkpp'])
        .reset_index(drop=True)
    )
    geometry_ranking.insert(0, 'rank_closest_to_fkpp', np.arange(1, len(geometry_ranking) + 1))

    geometry_ranking.to_csv(summary_folder / 'geometry_ranking_closest_to_fkpp.csv', index=False)
    display(geometry_ranking)

    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    labels = [fr'$L_y={row.Ly:g}$, $R={row.R_inter:g}$' for _, row in geometry_ranking.iterrows()]
    ax.barh(labels[::-1], geometry_ranking['mean_abs_error_to_fkpp'].to_numpy()[::-1])
    ax.set_xlabel(r'mean $|v_{sim}/v_{FKPP}-1|$ over $D_r$')
    ax.set_title('Geometry ranking: closest to FKPP')
    ax.grid(axis='x', alpha=0.25)
    fig.tight_layout()

    save_path = figures_folder / 'geometry_ranking_closest_to_fkpp.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 9. Fit-window sensitivity / stationarity check

In [ ]:
all_fit_rows = []
all_window_summaries = []

for f0, f1 in fit_windows:
    label = window_label(f0, f1)
    fit_df_w, missing_w = fit_runs_for_window(run_table, f0, f1)
    print(f'Window {label}: fitted {len(fit_df_w)} / {len(run_table)} runs')
    if missing_w:
        print('  Missing examples:', missing_w[:10], '...' if len(missing_w) > 10 else '')

    if len(fit_df_w) == 0:
        continue

    run_level_w = make_run_level_speed_table(fit_df_w, group_cols)
    summary_w = summarize_run_level(run_level_w, group_cols)
    summary_w['fit_window'] = label
    summary_w['fit_start_fraction'] = f0
    summary_w['fit_end_fraction'] = f1

    fit_df_w['fit_window'] = label
    all_fit_rows.append(fit_df_w)
    all_window_summaries.append(summary_w)

if len(all_window_summaries) > 0:
    window_summary = pd.concat(all_window_summaries, ignore_index=True)
    all_window_fits = pd.concat(all_fit_rows, ignore_index=True)
else:
    window_summary = pd.DataFrame()
    all_window_fits = pd.DataFrame()

window_summary.to_csv(summary_folder / 'fit_window_summary.csv', index=False)
all_window_fits.to_csv(summary_folder / 'fit_window_all_fits.csv', index=False)

display(window_summary.head(20))

## 10. Fit-window sensitivity plot: $v_{sim}/v_{FKPP}$

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary to plot yet.')
else:
    nrows = len(fit_windows)
    ncols = len(Ly_values)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.0 * ncols, 2.9 * nrows),
        sharex=True,
        sharey=True,
    )

    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = np.array([axes])
    elif ncols == 1:
        axes = np.array([[ax] for ax in axes])

    for row_idx, (f0, f1) in enumerate(fit_windows):
        label = window_label(f0, f1)

        for col_idx, Ly in enumerate(Ly_values):
            ax = axes[row_idx, col_idx]
            sub_Ly = window_summary[
                (window_summary['fit_window'] == label) &
                np.isclose(window_summary['Ly'], Ly)
            ]

            for R in R_inter_values:
                sub = sub_Ly[np.isclose(sub_Ly['R_inter'], R)].sort_values('Dr')
                if len(sub) == 0:
                    continue
                ax.errorbar(
                    sub['Dr'],
                    sub['mean_speed_over_fkpp'],
                    yerr=sub['speed_over_fkpp_sem'].fillna(0.0),
                    marker='o',
                    linestyle='-',
                    capsize=3,
                    linewidth=1.2,
                    label=fr'$R={R:g}$',
                )

            ax.axhline(1.0, linestyle='--', linewidth=1.0)
            ax.grid(alpha=0.25)
            if row_idx == 0:
                ax.set_title(fr'$L_y={Ly:g}$')
            if col_idx == 0:
                ax.set_ylabel(f'fit {label}\n' + r'$v_{sim}/v_{FKPP}$')
            if row_idx == nrows - 1:
                ax.set_xlabel(r'$D_r$')

    axes[0, -1].legend(fontsize=9, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.suptitle(fr'Fit-window sensitivity, glued $B={isolation_buffer_factor:g}R_{{inter}}$', y=1.01)
    fig.tight_layout()

    save_path = figures_folder / 'glued_brownian_v_over_fkpp_fit_window_sensitivity.png'
    fig.savefig(save_path, dpi=180, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 11. Fit-window spread summary

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary yet.')
else:
    window_spread = (
        window_summary
        .groupby(['Ly', 'R_inter', 'Dr'], as_index=False)
        .agg(
            min_v_over_fkpp=('mean_speed_over_fkpp', 'min'),
            max_v_over_fkpp=('mean_speed_over_fkpp', 'max'),
            mean_v_over_fkpp=('mean_speed_over_fkpp', 'mean'),
            std_v_over_fkpp_across_windows=('mean_speed_over_fkpp', 'std'),
            n_windows=('mean_speed_over_fkpp', 'count'),
        )
    )
    window_spread['range_v_over_fkpp_across_windows'] = (
        window_spread['max_v_over_fkpp'] - window_spread['min_v_over_fkpp']
    )
    window_spread['abs_mean_error_to_fkpp'] = np.abs(window_spread['mean_v_over_fkpp'] - 1.0)

    window_spread = window_spread.sort_values(
        ['range_v_over_fkpp_across_windows', 'abs_mean_error_to_fkpp']
    ).reset_index(drop=True)

    window_spread.to_csv(summary_folder / 'fit_window_spread_stationarity_ranking.csv', index=False)
    display(window_spread.head(30))

    geometry_stationarity = (
        window_spread
        .groupby(['Ly', 'R_inter'], as_index=False)
        .agg(
            mean_window_range=('range_v_over_fkpp_across_windows', 'mean'),
            max_window_range=('range_v_over_fkpp_across_windows', 'max'),
            mean_abs_error_to_fkpp=('abs_mean_error_to_fkpp', 'mean'),
        )
        .sort_values(['mean_window_range', 'mean_abs_error_to_fkpp'])
        .reset_index(drop=True)
    )
    geometry_stationarity.insert(0, 'rank_most_stationary', np.arange(1, len(geometry_stationarity) + 1))

    geometry_stationarity.to_csv(summary_folder / 'geometry_stationarity_ranking.csv', index=False)
    display(geometry_stationarity)

## 12. Compact final table

In [ ]:
if len(summary) == 0 or len(window_summary) == 0:
    print('Need both main summary and fit-window summary.')
else:
    final_geometry_table = geometry_ranking.merge(
        geometry_stationarity,
        on=['Ly', 'R_inter'],
        how='left',
        suffixes=('_fkpp', '_stationarity'),
    )

    final_geometry_table = final_geometry_table[[
        'Ly', 'R_inter',
        'rank_closest_to_fkpp',
        'mean_abs_error_to_fkpp_fkpp',
        'max_abs_error_to_fkpp',
        'mean_signed_error_to_fkpp',
        'mean_v_over_fkpp',
        'rank_most_stationary',
        'mean_window_range',
        'max_window_range',
    ]].sort_values(['rank_closest_to_fkpp', 'rank_most_stationary'])

    final_geometry_table.to_csv(summary_folder / 'final_geometry_choice_table.csv', index=False)
    display(final_geometry_table)

## Threshold comparison

In [ ]:
# ==========================================================
# Threshold comparison summary for Dr sweep
# Compares th_1, th_2, th_3 as v_front(Dr)
# ==========================================================

threshold_methods = ["th_1", "th_2", "th_3"]

threshold_label_map = {
    "th_1": rf"${base['threshold_frac1']:g}\rho_{{\mathrm{{sat}}}}$",
    "th_2": rf"${base['threshold_frac2']:g}\rho_{{\mathrm{{sat}}}}$",
    "th_3": rf"${base['threshold_frac3']:g}\rho_{{\mathrm{{sat}}}}$",
}

threshold_run_rows = []

keep_cols = ["run_id", "seed"] + group_cols + ["v_fkpp"]
keep_cols = [c for c in keep_cols if c in fit_df.columns]

for method in threshold_methods:
    speed_long = side_speed_long_table(
        fit_df,
        method=method,
        keep_columns=keep_cols,
    )

    if len(speed_long) == 0:
        print(f"No data found for {method}. Check that fit_df contains left/right_{method}_speed.")
        continue

    speed_long = speed_long.replace([np.inf, -np.inf], np.nan)
    speed_long = speed_long.dropna(subset=["speed"]).copy()

    if "v_fkpp" in speed_long.columns:
        speed_long["speed_over_fkpp"] = speed_long["speed"] / speed_long["v_fkpp"]

    # First average left/right fronts within each run
    run_cols = ["run_id"] + group_cols

    agg_dict = {
        "run_speed": ("speed", "mean"),
    }

    if "speed_over_fkpp" in speed_long.columns:
        agg_dict["run_speed_over_fkpp"] = ("speed_over_fkpp", "mean")

    if "v_fkpp" in speed_long.columns:
        agg_dict["v_fkpp"] = ("v_fkpp", "first")

    run_level_m = (
        speed_long
        .groupby(run_cols, as_index=False)
        .agg(**agg_dict)
    )

    run_level_m["method"] = method
    threshold_run_rows.append(run_level_m)

if len(threshold_run_rows) == 0:
    threshold_run_level = pd.DataFrame()
    threshold_summary = pd.DataFrame()
    print("No threshold data found.")
else:
    threshold_run_level = pd.concat(threshold_run_rows, ignore_index=True)

    threshold_summary = (
        threshold_run_level
        .groupby(group_cols + ["method"], as_index=False)
        .agg(
            n_runs=("run_speed", "count"),
            mean_speed=("run_speed", "mean"),
            speed_std=("run_speed", "std"),
        )
    )

    threshold_summary["speed_sem"] = (
        threshold_summary["speed_std"] / np.sqrt(threshold_summary["n_runs"])
    )

    if "run_speed_over_fkpp" in threshold_run_level.columns:
        ratio_summary = (
            threshold_run_level
            .groupby(group_cols + ["method"], as_index=False)
            .agg(
                mean_speed_over_fkpp=("run_speed_over_fkpp", "mean"),
                speed_over_fkpp_std=("run_speed_over_fkpp", "std"),
            )
        )

        threshold_summary = threshold_summary.merge(
            ratio_summary,
            on=group_cols + ["method"],
            how="left",
        )

        threshold_summary["speed_over_fkpp_sem"] = (
            threshold_summary["speed_over_fkpp_std"]
            / np.sqrt(threshold_summary["n_runs"])
        )

    display(threshold_summary.sort_values(["Ly", "R_inter", "method", "Dr"]))

In [ ]:
# ==========================================================
# Plot: v_front vs Dr for the three thresholds
# Choose one Ly and one R_inter
# ==========================================================

if len(threshold_summary) == 0:
    print("No threshold summary to plot yet.")
else:
    plot_Ly = 2.0
    plot_R = 0.05

    plot_df = threshold_summary[
        np.isclose(threshold_summary["Ly"], plot_Ly)
        & np.isclose(threshold_summary["R_inter"], plot_R)
    ].copy()

    if len(plot_df) == 0:
        print(f"No data for Ly={plot_Ly}, R_inter={plot_R}.")
    else:
        fig, ax = plt.subplots(figsize=(7.0, 4.5))

        for method in threshold_methods:
            sub = (
                plot_df[plot_df["method"] == method]
                .sort_values("Dr")
            )

            if len(sub) == 0:
                continue

            ax.errorbar(
                sub["Dr"],
                sub["mean_speed"],
                yerr=sub["speed_sem"].fillna(0.0),
                marker="o",
                linestyle="-",
                capsize=3,
                linewidth=1.5,
                label=threshold_label_map.get(method, method),
            )

        Dr_line = np.linspace(plot_df["Dr"].min(), plot_df["Dr"].max(), 300)
        r_edge_value = float(p0 - q0)

        ax.plot(
            Dr_line,
            2.0 * np.sqrt(Dr_line * r_edge_value),
            "--",
            linewidth=1.5,
            label="FKPP",
        )

        ax.set_xlabel(r"$D_r$")
        ax.set_ylabel(r"$v_{\mathrm{front}}$")
        ax.set_title(fr"Threshold comparison, $L_y={plot_Ly:g}$, $R={plot_R:g}$")

        ax.legend(frameon=False)
        style_axis(ax, n_ticks=4, grid=False)

        fig.tight_layout()
        save_path = figures_folder / f"threshold_comparison_vfront_vs_Dr_Ly{plot_Ly:g}_R{plot_R:g}.png"
        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)
        plt.show()

## Thesis plots

In [ ]:
from matplotlib.ticker import MaxNLocator

R_colors = {
    0.05: "tab:blue",
    0.08: "tab:orange",
    0.10: "tab:green",
}

color_fkpp = "tab:red"
color_fit = "0.55"   # grey

def get_R_color(R):
    for R_key, color in R_colors.items():
        if np.isclose(R, R_key):
            return color
    return None

In [ ]:
Ly_target = 2

if len(summary) == 0:
    print("No summary to plot yet.")
else:
    fig, ax = plt.subplots(figsize=(5.5, 4.2))

    sub_Ly = summary[np.isclose(summary["Ly"], Ly_target)]

    Dr_line = np.linspace(sub_Ly["Dr"].min(), sub_Ly["Dr"].max(), 300)
    v_fkpp_line = 2.0 * np.sqrt(Dr_line * r_edge)

    for R in R_inter_values:
        sub = sub_Ly[np.isclose(sub_Ly["R_inter"], R)].sort_values("Dr")
        if len(sub) == 0:
            continue

        ax.errorbar(
            sub["Dr"],
            sub["mean_speed"],
            yerr=sub["speed_sem"].fillna(0.0),
            marker="o",
            linestyle="-",
            capsize=3,
            linewidth=1.5,
            color=get_R_color(R),
            label=fr"$R={R:g}$",
        )

    ax.plot(
        Dr_line,
        v_fkpp_line,
        "--",
        linewidth=1.7,
        color=color_fkpp,
        label="FKPP",
    )

    ax.set_xlabel(r"$D_r$")
    ax.set_ylabel(r"$v_{\mathrm{front}}$")
    ax.grid(False)
    ax.legend(fontsize=14, frameon=False)

    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    fig.tight_layout()

    save_path = figures_folder / f"brownian_velocity_vs_Dr_Ly{Ly_target:g}.pdf"
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show()

In [ ]:
Ly_target = 2
R_target = 0.05

if len(summary) == 0:
    print("No summary to plot yet.")
else:
    fig, ax = plt.subplots(figsize=(5.5, 4.2))

    sub = summary[
        np.isclose(summary["Ly"], Ly_target)
        & np.isclose(summary["R_inter"], R_target)
    ].sort_values("Dr").copy()

    if len(sub) == 0:
        print(f"No data found for Ly={Ly_target} and R_inter={R_target}")
    else:
        x = sub["Dr"].to_numpy()
        y = (sub["mean_speed"] ** 2).to_numpy()

        yerr = (
            2.0
            * sub["mean_speed"]
            * sub["speed_sem"].fillna(0.0)
        ).to_numpy()

        Dr_line = np.linspace(x.min(), x.max(), 300)
        v_fkpp2_line = 4.0 * Dr_line * r_edge

        fit_coeffs = np.polyfit(x, y, deg=1)
        fit_slope, fit_intercept = fit_coeffs
        fit_line = fit_slope * Dr_line + fit_intercept

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            marker="o",
            linestyle="none",
            capsize=3,
            linewidth=1.5,
            color=get_R_color(R_target),
            label=fr" $R={R_target:g}$",
        )

        ax.plot(
            Dr_line,
            fit_line,
            "-",
            linewidth=1.7,
            color=color_fit,
            label="Linear fit",
        )

        ax.plot(
            Dr_line,
            v_fkpp2_line,
            "--",
            linewidth=1.7,
            color=color_fkpp,
            label="FKPP",
        )

        ax.set_xlabel(r"$D_r$")
        ax.set_ylabel(r"$v_{\mathrm{front}}^2$")
        ax.grid(False)
        ax.legend(fontsize=14, frameon=False)

        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

        fig.tight_layout()

        save_path = figures_folder / (
            f"brownian_velocity_squared_vs_Dr_"
            f"Ly{Ly_target:g}_R{str(R_target).replace('.', 'p')}.pdf"
        )
        fig.savefig(save_path, dpi=180, bbox_inches="tight")
        print("Saved:", save_path)
        print(f"Linear fit: v_front^2 = {fit_slope:.6g} D_r + {fit_intercept:.6g}")
        print(f"FKPP slope: 4r = {4.0 * r_edge:.6g}")

        plt.show()

In [ ]:
if len(summary) == 0:
    print("No summary to plot yet.")
else:
    Ly_plot_values = [0.5, 1, 2, 4]

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(9.5, 6.8),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()

    for ax, Ly in zip(axes, Ly_plot_values):
        sub_Ly = summary[np.isclose(summary["Ly"], Ly)]

        for R in R_inter_values:
            sub = sub_Ly[np.isclose(sub_Ly["R_inter"], R)].sort_values("Dr")
            if len(sub) == 0:
                continue

            ax.errorbar(
                sub["Dr"],
                sub["mean_speed_over_fkpp"],
                yerr=sub["speed_over_fkpp_sem"].fillna(0.0),
                marker="o",
                linestyle="-",
                capsize=3,
                linewidth=1.5,
                color=get_R_color(R),
                label=fr"$R={R:g}$",
            )

        ax.axhline(
            1.0,
            linestyle="--",
            linewidth=1.5,
            color=color_fkpp,
            label="FKPP",
        )

        ax.set_title(fr"$L_y={Ly:g}$")
        ax.grid(False)

        ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4))

    axes[2].set_xlabel(r"$D_r$")
    axes[3].set_xlabel(r"$D_r$")
    axes[0].set_ylabel(r"$v_{\mathrm{front}}/v_{\mathrm{FKPP}}$")
    axes[2].set_ylabel(r"$v_{\mathrm{front}}/v_{\mathrm{FKPP}}$")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        fontsize=14,
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=4,
    )

    fig.tight_layout(rect=[0, 0, 1, 0.95])

    save_path = figures_folder / "brownian_v_over_fkpp_by_Ly_Rinter_2x2.pdf"
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    print("Saved:", save_path)

    plt.show()